# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'social profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [17]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community/forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Careers', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-H3
Updated
about 14 hours ago
•
12.1k
•
2.79k
deepseek-ai/DeepSeek-V4-Flash-0731
Updated
6 days ago
•
618k
•
2.67k
moonshotai/Kimi-K3
Updated
11 days ago
•
1.26M
•
10.2k
Comfy-Org/MiniMax-H3
Updated
1 day ago
•
2.3M
•
863
unsloth/DeepSeek-V4-Flash-0731-GGUF
Updated
a

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-H3\nUpdated\nabout 14 hours ago\n•\n12.1k\n•\n2.79k\ndeepseek-ai/DeepSeek-V4-Flash-0731\nUpdated\n

In [ ]:
system_message = "You are a helpful assistant."
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

def brochure_with_model(company, URL):
    prompt = f"Please generate a company brochure for {company}. Here is their landing page:\n"
    prompt += fetch_website_contents(URL)
    results = stream_gpt(prompt)
    yield from results

In [33]:
import gradio as gr
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")

message_output = gr.Textbox(label="Response:")

view = gr.Interface(
    fn=brochure_with_model,
    title="What company do you want to get a brochure for?",
    inputs = [message_input, url_input],
    outputs = [message_output],
    examples=[["HuggingFace", "https://huggingface.co"]],
    flagging_mode="never"
)
view.launch()


* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\sfkei\projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sfkei\projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sfkei\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sfkei\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sfkei\projects\llm_engineering\.venv\Lib\site-packages\gradio\utils.py", line

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It is a collaboration platform where the global ML community creates, discovers, and collaborates on models, datasets, and applications. Hugging Face hosts over **2 million models**, **500,000 datasets**, and **over 1 million applications**, making it a world-leading hub for machine learning innovation.

---

## Our Mission

To empower the machine learning community through open collaboration, enabling faster progress and democratizing AI technology worldwide.

---

## What We Offer

### 1. Models  
Explore a vast repository of over 2 million pre-trained AI models covering diverse tasks such as natural language processing, computer vision, speech recognition, and more. The platform highlights trending and newly updated models constantly contributed and improved by the global community.

### 2. Datasets  
Access and share over 500,000 datasets essential for ML training, fine-tuning, and benchmarking. Our datasets span multiple domains, ensuring that data scientists and researchers have the resources they need.

### 3. Spaces  
Run and share interactive AI applications easily with Hugging Face Spaces. These demo-ready environments allow users to experiment and visualize AI capabilities through intuitive interfaces.

### 4. Buckets and Storage  
Secure and scalable storage solutions for hosting models, datasets, and applications, enabling seamless delivery and deployment.

### 5. Enterprise Solutions  
Tailored enterprise support including Hugging Face PRO, team collaboration tools, enterprise-grade inference endpoints, and dedicated support cater to business needs accelerating AI adoption.

---

## Community & Culture

Hugging Face thrives on an open, collaborative ethos where knowledge sharing and mutual support propel the AI revolution. The company fosters a global community of ML practitioners, researchers, engineers, and enthusiasts who contribute to and benefit from a shared platform.

The community is active via multiple channels including:  
- Discord  
- Forum  
- GitHub  
- Blog and Daily Papers insights  

Culture highlights:  
- Openness and transparency  
- Innovation driven by collaboration  
- Commitment to democratizing AI  
- Inclusive environment for learners and experts alike

---

## Customers & Use Cases

Hugging Face serves a broad spectrum of users:  
- **AI Researchers & Data Scientists** leveraging rapid access to state-of-the-art models and datasets  
- **Developers & Engineers** building AI-powered applications with robust APIs and deployment tools  
- **Enterprises** seeking scalable AI infrastructure and support for integrating ML into business solutions  
- **Academic institutions** using the platform for teaching and research

Users benefit from scalable inference endpoints, collaborative spaces, and enterprise-grade tools to accelerate AI development and deployment.

---

## Careers at Hugging Face

Join a company that is shaping the future of AI through community-driven innovation. Hugging Face offers opportunities for talented individuals passionate about machine learning, software engineering, research, product management, and AI infrastructure.

Employees enjoy:  
- A purpose-driven mission at the forefront of ML  
- Dynamic and collaborative company culture  
- Opportunities to work with cutting-edge technologies  
- Access to a vibrant, global AI community

Check the latest openings to be part of a fast-growing AI company dedicated to open collaboration and democratizing machine learning.

---

## Get Involved

- **Explore models, datasets, and apps:** https://huggingface.co  
- **Join the community on Discord and Forums**  
- **Contribute on GitHub**  
- **Follow blogs and daily AI papers** to stay current with AI research trends

---

## Contact & Follow

- Website: [https://huggingface.co](https://huggingface.co)  
- Community: Discord, Forum, GitHub  
- Blog: Updates on research, products, and AI trends  

---

**Hugging Face — The AI community building the future.**  
Collaborate. Innovate. Democratize AI.

---

*Brand colors:*  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face  
Hugging Face is a pioneering AI company and an open collaboration platform dedicated to advancing the field of machine learning. As "The AI community building the future," it provides a vibrant space where the global machine learning community comes together to create, share, and evolve models, datasets, and applications that power the next generation of AI technologies.

---

## What We Offer  

- **Models:** Browse and collaborate on over 2 million machine learning models, ranging from natural language processing to computer vision and beyond.  
- **Datasets:** Access over 500,000 datasets to fuel your AI projects, updated and maintained by a diverse, global community.  
- **Spaces:** Host and explore AI-powered applications in a community-driven environment, featuring interactive demos and tools.  
- **Buckets:** Storage solutions to host and manage datasets and models efficiently.  
- **HuggingChat:** Engage with conversational AI directly via Hugging Face’s chat platform.  
- **Enterprise Solutions:** Tailored offerings for organizations including team collaboration tools, enterprise support, inference endpoints, and PRO accounts.

---

## Community & Collaboration  
Hugging Face champions an open, collaborative culture where thousands of researchers, developers, and enterprises share their work openly on the Hugging Face Hub. This collaborative spirit empowers innovation by making it easy to discover, use, and build upon state-of-the-art machine learning resources. The community engagement extends to active forums, Discord servers, GitHub repositories, blogs, and daily papers to stay at the cutting edge of AI research.

---

## Customers & Users  
- **Developers & Researchers:** Access state-of-the-art tools and models to accelerate AI research and development.  
- **Enterprises:** Benefit from scalable AI infrastructure, enterprise-grade support, and custom solutions to embed AI in business workflows.  
- **Educators & Students:** Utilize vast curated datasets and models to learn, teach, and build AI projects.  
- **AI Enthusiasts & Innovators:** Explore new AI applications and contribute to open-source projects.

---

## Company Culture  
Hugging Face fosters an inclusive, open, and innovative environment. The core values emphasize transparency, collaboration, and community-driven progress. Employees and community members work hand in hand to democratize AI technology, making it accessible to all while striving for excellence in machine learning innovation.

---

## Career Opportunities  
Hugging Face is continuously growing and seeking passionate professionals in fields such as:  
- Machine Learning Research and Engineering  
- Software Development  
- Data Science and Analytics  
- Community Management and Developer Relations  
- Enterprise Sales and Customer Success  

Joining Hugging Face means becoming part of a mission-driven team focused on building open and powerful AI tools that impact the future.

---

## Connect With Us  
- **Website:** [huggingface.co](https://huggingface.co)  
- **Community Forums:** Engage with developers and AI experts  
- **Discord Server:** Join real-time discussions and collaborations  
- **GitHub:** Explore code repositories and contribute  
- **Blog & Daily Papers:** Stay updated on AI trends and announcements  

---

Hugging Face is not just a company; it’s a global movement powering the next wave of AI innovation through open collaboration.  


**Join us and build the future of AI together!**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>